## **CUDA Runtime**

- Copies the data from the host to the device

- Load GPU kernel and execute it

- Copy the result from the device to the host

<hr>

## **Device Vs Host Naming Scheme**

- `h_Var` for host variable

- `d_Var` for device variable

**`__global__`**

-  It is a CUDA C/C++ keyword used to declare a function as a kernel function that can be executed on the GPU.

- A kernel function is a function that is executed on the GPU and can be called from the host (CPU) code.

**`__device__`**

- It is a CUDA C/C++ keyword used to declare a function or variable as a device function or variable that can only be accessed and executed on the GPU.

- A device function is a function that can only be called from other device functions or kernel functions, and cannot be called from host code.

**`__host__`**

- It is a CUDA C/C++ keyword used to declare a function as a host function that can only be executed on the CPU.

- A host function is a function that can only be called from other host functions or kernel functions, and cannot be called from device code.

<hr>

## **Memory Management**

**`cudaMalloc`**

- It is a CUDA C/C++ function used to allocate memory on the GPU device.

- The syntax for `cudaMalloc` is as follows:

```cpp
cudaError_t cudaMalloc(void **devPtr, size_t size);
```

**`cudaMemcpy`**

- It is a CUDA C/C++ function used to copy data between device -> device (One GPU location to another), host -> device, and device -> host.

- The syntax for `cudaMemcpy` is as follows:

```cpp
cudaError_t cudaMemcpy(void *dst, const void *src, size_t count, cudaMemcpyKind kind);
```

**`cudaFree`**

- It is a CUDA C/C++ function used to free memory that was previously allocated on the GPU device using `cudaMalloc`.

- The syntax for `cudaFree` is as follows:

```cpp
cudaError_t cudaFree(void *devPtr);
```

<hr>

The pipeline of compiling and executing a CUDA program relies on a sophisticated "split-compilation" model. To understand how the NVIDIA CUDA Compiler (`nvcc`), PTX, JIT compilation, and forward compatibility work together, we can break the process down into five sequential steps.

---

### 1. NVCC and Split Compilation (Host vs. Device Code)
CUDA code is heterogeneous; a single `.cu` source file contains both **host code** (which runs on the CPU) and **device code** (the kernels that run on the GPU). 

Because CPUs and GPUs have entirely different instruction set architectures (ISAs) and memory spaces, they cannot execute the same binary. `nvcc` acts as a compiler driver to solve this:

1. **Code Splitting:** `nvcc` parses the `.cu` file and separates the CPU code from the GPU kernels (demarcated by `__global__` and `__device__` keywords).
2. **Host Code Modification:** Standard CPU compilers (like GCC on Linux or MSVC on Windows) cannot understand CUDA’s custom kernel launch syntax (the triple-angle brackets: `kernel<<<blocks, threads>>>(...)`). `nvcc` modifies this host code, replacing the `<<<...>>>` syntax with standard host-compatible C++ Runtime API calls (such as `cudaLaunchKernel`).
3. **Host Compilation:** This modified host code is passed to the host compiler, which compiles it into a standard host binary (such as an **x86 binary** on standard Intel/AMD systems). The resulting CPU executable is responsible for managing memory allocations, copying data to the GPU, and launching the GPU kernels using those inserted runtime API calls.

---

### 2. PTX (Parallel Thread Execution) – The Virtual ISA
While the host code is compiled to native CPU instructions (like x86), the device code (the GPU kernels) undergoes its own compilation path. Instead of compiling directly to the machine code of a specific GPU, `nvcc` can compile the kernels into **PTX (Parallel Thread Execution)**.

* **What is PTX?** PTX is a low-level, virtual machine instruction set architecture (ISA) designed by NVIDIA. It resembles assembly language but targets a *standardized, virtual GPU* rather than physical hardware.
* **Abstraction:** PTX abstracts away physical hardware limits, such as the exact number of registers, warp schedulers, or memory layout of a specific physical GPU model. It provides a stable intermediate representation (IR).

---

### 3. Stability Across GPU Generations
NVIDIA releases new GPU architectures regularly (e.g., Pascal, Turing, Ampere, Ada Lovelace, Hopper, Blackwell). Each of these generations features a different **real architecture** with its own native, physical instruction set, known as **SASS (Source Assembly)**. 

Because SASS is highly optimized for the physical layout of a specific chip generation, a binary compiled directly to SASS for an older GPU (e.g., Ampere) will not run on a newer GPU (e.g., Blackwell) if the underlying hardware instructions or register architectures have changed.

PTX solves this by being **stable across multiple GPU generations**. While physical SASS changes radically between architectures, the virtual PTX ISA is backward- and forward-compatible. PTX instructions compiled years ago remain valid and interpretable by newer CUDA drivers.

---

### 4. Just-In-Time (JIT) Compilation at Runtime
When you compile a CUDA program, `nvcc` packages both the compiled host code (the x86 binary) and the device code together into a single executable called a **fat binary**. The device code embedded in this fat binary can contain:
1. **SASS:** Native instructions pre-compiled for specific physical GPUs (e.g., `sm_80` for Ampere).
2. **PTX:** Virtual instructions for a base virtual architecture (e.g., `compute_80`).

When the application runs, the **CUDA Driver** (installed on the host system) steps in to manage kernel execution. 

* If the system's current physical GPU matches one of the pre-compiled SASS configurations in the fat binary, the driver loads that SASS directly to the GPU for maximum speed.
* If the physical GPU is newer than any of the pre-compiled SASS configurations, the driver extracts the embedded PTX code. 
* The driver's built-in compiler performs **Just-In-Time (JIT) compilation**, translating the virtual **PTX instructions into the native SASS instructions** of the host's specific GPU on the fly.
* To prevent performance overhead on subsequent runs, the compiled SASS is saved to a system-wide **JIT cache** on the host's storage.

---

### 5. Achieving Forward Compatibility
This split compilation and JIT model is what **allows for forward compatibility**.

Without PTX and JIT compilation, you would have to recompile your CUDA code every time NVIDIA released a new GPU architecture. By embedding PTX inside your compiled binary, your software is future-proofed. 

```
[ .cu Source File ] 
       │
       ▼ (nvcc splits the code)
       ├─────────────────────────────────────────┐
       ▼ (Host Path)                             ▼ (Device Path)
[ Modified Host Code ]                     [ GPU Kernels ]
       │                                         │
       ▼ (compiled by GCC/MSVC)                  ▼ (compiled by nvcc)
[ x86 Host Binary ]                        [ PTX (Virtual ISA) ]
       │                                         │
       └──────────────────┬──────────────────────┘
                          ▼ (Linked together)
                  [ Fat Binary (.exe) ]
                          │
         ┌────────────────┴────────────────┐ (At Runtime)
         ▼ (Old/Target GPU)                ▼ (New GPU Generation)
  Runs pre-compiled SASS              CUDA Driver JIT Compiles:
                                    [ PTX ] ──► [ New Native SASS ]
```

When a user runs your older application on a brand-new, future GPU, the application doesn't crash. The updated CUDA driver on the user’s system simply JIT-compiles the stable PTX code into the new GPU's physical instruction set, allowing the application to run seamlessly on hardware that did not even exist when the code was written.